## 생성형 모델

데이터의 분포를 학습하여, 존재하지 않던 새로운 데이터를 만들어내는 모델

- 확률적 아이디어

숫자 3을 써도 사람마다 모양, 굵기, 기울기가 다르므로 이미지는 확률적인 분포로 표현할 수 있다.

따라서 데이터가 만들어졌을 가능성 공간(Latent Space)를 학습하자!

## GAN

생성적 적대 신경망으로 생성기와 판별기라는 두 개의 신경망이 경쟁적으로 학습하여 이미지를 생성하는 모델

생성기는 진짜처럼 보이는 가짜 데이터를 생성하고

판별기는 입력된 데이터가 진짜인지 가짜인지 구별하는 역할을 한다.

- GAN의 특징 및 문제

불안정한 학습 : loss가 줄어들지 않고 계속 진동하는 현상이 빈번하다.

모드 붕괴 : G가 다양성을 잃고, D를 속이기 쉬운 특정 이미지만 반복 생성한다.

### GAN의 수학적 의미

D(x): 진짜 이미지에 대한 판별기의 출력

D(G(z)): 가짜 이미지에 대한 판별기의 출력

학습은 교차 최적화 방식으로 진행되며 '최소 최대 게임' (minimax two-player game)형태로 표현된다.

min_G max_D V(D, G) = E[log D(x)] + E[log(1 − D(G(z)))]


## DCGAN

기존의 GAN에 CNN구조를 체계적으로 도입하여 안정적인 학습과 고품질 이미지 생성을 가능하게한, 이후 거의 모든 GAN 변형의 기반이 된 모델



### DCGAN의 핵심 아키텍처

CNN 기반 GAN을 안정적으로 학습시키기 위한 설계 원칙 정립

- Pooling 제거

Max Pooling 대신 생성자에서는 전치 합성곱, 또는 분수 스트라이드 합성곱을,

판별자에서는 스트라이드 합성곱을 사용하여 네트워크가 스스로 다운샘플링 / 업샘플링 방법을 학습하게 된다.

- 배치 정규화 적용

생성자와 판별자 모두에 배치 정규화를 적용한다.

단, 생성자의 출력층과 판별자의 입력층에는 적용하지 않는다.

이를 통해 학습 초기의 불안정성과 mode collapse를 크게 줄였다.

- 완전 연결층 제거

히든 레이어에서 FC를 전부 없애고 전부 합성곱 연산으로 대체한다.

- 활성화 함수 선택

생성자에서는 출력층을 제외한 모든 레이어에 ReLU를 사용하고, 출력층에만 Tanh를 적용한다.

판별자에서는 모든 레이어에 LeakyReLU를 사용한다.



### DCGAN 생성자 구조

생성자는 랜덤 노이즈 벡터z을 입력받아 이미지를 생성한다.

입력 : 64차원 노이즈 벡터를 선형 계층을 통해 16x16x128 특징맵으로 변환.

업샘플링 1단계 : 16x 16 -> 32x32 (합성곱 계층 사용)

업샘플링 2단계 : 32x32 -> 64x64 (추가 합성곱 계층 적용)

각 단계마다 3x3 커널, 128채널을 사용해 점차 해상도를 높인다.

마지막에 2차원 합성곱 계층을 거쳐 64x64 RGB 이미지를 생성한다.

핵심 : 노이즈로부터 점진적으로 해상도를 높이며 현실적인 이미지를 생성하는 구조

### DCGAN 판별자 구조

판별자는 생성자의 거울 구조로, 이미지를 입력받아 진짜/가짜를 판별한다.

입력 : 64x64 (3채널) RGB 이미지를 입력받는다.

첫 합성곱 계층 : 커널 크기 3x3, 스트라이드 2, 64채널의 특징맵 생성

두 번째 합성곱 : 128채널, 32x32 -> 16x17

세 번째 합성곱 : 256채널, 16x16 -> 8x8

네 번째 합성곱 : 512채널, 8x8 -> 4x4


마지막 단계

전결합층(FC)로 연결

시그모이드 함수를 통해 이미지가 진짜(1)인지 가짜(0)인지 판별한다.

핵심 : 판별기는 합성곱과 다운샘플링을 반복해 입력 이미지의 특징을 추출하고 최종적으로 진짜/가짜 확률을 출력하는 이진 분류기 역할을 수행한다.

## Pix2Pix GAN

단순한 이미지 생성이 아니라 이미지 -> 이미지 변환 작업에 특화된 모델로

쌍을 이루는 이미지 데이터를 기반으로, 한 도메인의 이미지를 다른 도메인의 이미지로 변환하는 범용 프레임워크이다.

스타일 전이, 색 변환, 스케치-> 사진 변환 등 layer간 mapping학습을 수행한다.

### Pix2Pix 생성자 구조

U-Net 기반의 Encoder-Decoder 형태

- 인코더(다운샘플링)

1. 8단게 2차원 합성곱 계층으로 구성.

2. 각 계층 : 합성곱 -> 정규화 -> LeakyReLU활성화

3. 입력 이미지를 점진적으로 축소하여 특징 추출

- 디코더(업샘플링)

1. 업샘플링 + 합성곱 계층으로 구성

2. 각 단계에서 skip connection으로 인코더 출력과 연결(concat)

3. 활성화 함수 : ReLU, 마지막 단계는 tanh

- 출력

1. 입력 이미지와 동일 크기의 변환 이미지 생성

2. 이미지 간 변화(예 : 흑백 -> 컬러)에 활용.

핵심 : 인코더로 압축된 특징을 디코더로 복원하며, skip connection을 통해 세밀한 구조를 유지하는 U-Net형 생성기임

### Pix2Pix 판별자 구조

PatchGAN 기반의 CNN 판별기

- 입력

실제 이미지와 생성된 이미지를 나란히 결합(concat)하여 입력

- 합성곱 계층

여러 층의 conv + BatchNorm + LeakyReLU 구성. 입력 이미지를 70x70 크기의 작은 패치(patch)단위로 구분해 판별

- 출력

각 패치별로 진짜/가짜 확률을 예측(전체 평균이 최종 판별 결과)

핵심 : Patch GAN은 이미지 전체가 아니라 국소 패치 단위로 진위 판단을 수행하여 세밀한 질감을 효과적으로 학습시킴

## VAE

이미지를 잠재 공간(latent space)의 확률 분포로 변환하여 새로운 데이터를 생성하는 모델

- 기본 개념

입력 데이터를 잠재 공간에 확률적으로 인코딩하고 다시 복원하는 생성 모델이다.

일반 오토 인코더는 입력 x를 하나의 고정된 벡터 z로 매핑하지만 VAE의 인코더는 입력 x를 확률 분포의 파라미터(평균, 분산)으로 매핑한다.

### VAE 개념

1. 인코더

입력 데이터를 평균과 표준편차로 압축함

잠재 변수 z는 재매개변수화 트릭으로 샘플링함 z - u + σ * 𝜀

2. 잠재공간

z는 데이터의 압축된 확률적 표현으로, 새로운 데이터 생성에 사용됨

3. 디코더

z를 입력을 받아 원본과 유사한 데이터를 복원함.

일반적으로 sigmoid 활성화를 사용하여 [0,1] 범위의 이미지를 출력함